# Run COMPASS survival pipeline locally (PROFILE_data_processing sources)

Same pipeline, same two ICD-C61 cohort arms (**ARPI** and **ADT**), same
landmarks as `COMPASS_run_locally.ipynb`. The only difference is where the raw
clinical data comes from:

| | `COMPASS_run_locally.ipynb` | this notebook |
|---|---|---|
| source | one release, `OncDRS/ALL_2025_03/*.csv` | seven merged releases, `PROFILE_DATA/*.parquet` |
| output root | `data/CAIA/COMPASS/` | `data/CAIA/COMPASS_PROFILE_DATA/` |

The parquets are produced by the sibling `PROFILE_data_processing` repo
(`compile_OncDRS_data.ipynb`), which folds every OncDRS release pull
(`ALL_2021_11` ... `ALL_2026_03`) into one deduplicated, column-projected
parquet per table. Every path this notebook writes to lives under its own root,
so the existing `data/CAIA/COMPASS/` results are never touched and the two runs
can be compared side by side (final cell).

The pipeline scripts read either format transparently --
`data_preprocessing_common/oncdrs_sources.scan_source` casts parquet columns
back to the all-Utf8 schema `pl.scan_csv(..., infer_schema_length=0)` produces,
so nothing downstream of the read behaves differently.

Cells, in order:
0. **Schema audit** -- fails fast if a required column is absent or all-null in
   the merged parquets. This is the guard against the upstream `COLUMN_MAP`
   silently null-filling a renamed column (see `PROFILE_data_processing`);
   a null `NCI_PREFERRED_MED_NM` or `DERIVED_LAST_ALIVE_DATE` would otherwise
   produce an empty-but-plausible cohort rather than an error.
1. `compile_COMPASS_cohort_data.py` -- ADT-entry eligible cohort plus the ARPI-
   and ADT-anchored survival outputs.
2. `longitudinal_data_processing.py` -- once per anchor.
3. `build_prediction_inputs.py` + models, once per arm.
4. Summary.
5. **Cohort comparison** -- per-stage n deltas against the `ALL_2025_03` run.

Run the cells in order.


## Shared setup

In [ ]:
import json
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import polars as pl

PROJECT_ROOT = Path("/data/gusev/USERS/jpconnor/code/CAIA")
SURVIVAL_DIR = PROJECT_ROOT / "COMPASS" / "survival_analysis"
DATA_PREPROCESSING_DIR = PROJECT_ROOT / "COMPASS" / "data_preprocessing"

sys.path.insert(0, str(PROJECT_ROOT))
from data_preprocessing_common.oncdrs_sources import resolve as resolve_source
from data_preprocessing_common.oncdrs_sources import scan_source

# ---------------------------------------------------------------------------
# Data source: merged, deduplicated OncDRS parquets written by the sibling
# PROFILE_data_processing repo (compile_OncDRS_data.ipynb -> OUTPUT_PATH).
# resolve() picks the parquet when it exists under this root and falls back to
# the raw-release CSV basename otherwise, so a missing table surfaces in the
# audit below as a missing file rather than as an empty scan.
# ---------------------------------------------------------------------------
PROFILE_DATA_ROOT = Path("/data/gusev/USERS/jpconnor/data/PROFILE_DATA")
SOURCE_TABLES = [
    "EHR_DIAGNOSES",
    "MEDICATIONS",
    "LABS",
    "HEALTH_HISTORY",
    "PT_INFO_STATUS_REGISTRATION",
]
SOURCES = {t: resolve_source(t, PROFILE_DATA_ROOT) for t in SOURCE_TABLES}

# ---------------------------------------------------------------------------
# Output root. Deliberately NOT data/CAIA/COMPASS/ -- that root holds the
# ALL_2025_03 run this one is compared against, and nothing here may overwrite
# it. BASELINE_PROJ_PATH is read-only in this notebook (final comparison cell).
# ---------------------------------------------------------------------------
NEPC_PROJ_PATH = Path("/data/gusev/USERS/jpconnor/data/CAIA/COMPASS_PROFILE_DATA/")
BASELINE_PROJ_PATH = Path("/data/gusev/USERS/jpconnor/data/CAIA/COMPASS/")

MRN_LISTS_DIR = NEPC_PROJ_PATH / "mrn_lists"
DATA_ARPI = NEPC_PROJ_PATH / "longitudinal_prediction_data.csv"
DATA_ADT = NEPC_PROJ_PATH / "longitudinal_prediction_data_adt.csv"
CACHE_ARPI = NEPC_PROJ_PATH / "consolidated_longitudinal_data.parquet"
CACHE_ADT = NEPC_PROJ_PATH / "consolidated_longitudinal_data_adt.parquet"
SURVIVAL_OUTPUT_ROOT = NEPC_PROJ_PATH / "survival_analysis"

PYTHON = sys.executable
N_FOLDS = 5
FORCE_RERUN = True
RUN_GAM = True

# COMPASS durations (t_lab, t_platinum, ...) are measured from each arm's
# treatment anchor (time 0), so anchor_col is "none" for every arm below: the
# landmark is a pure offset from the anchor with no anchor column. Arms
# differ in (a) which anchor's Stage 1 survival cohort CSV restricts them
# (--restrict-to-mrns), (b) which anchor's Stage 2 output feeds them, and
# (c) both arms use landmarks [0, 90, 180]. Identical to COMPASS_run_locally.ipynb.
COHORT_SPECS = {
    "arpi": dict(
        anchor="arpi", landmarks=[0, 90, 180],
        mrn_csv=NEPC_PROJ_PATH / "prostate_arpi_survival_cohort_arpi.csv",
        title="ARPI",
    ),
    "adt": dict(
        anchor="adt", landmarks=[0, 90, 180],
        mrn_csv=NEPC_PROJ_PATH / "prostate_adt_survival_cohort_adt.csv",
        title="ADT",
    ),
}

# ARPI temporarily disabled: build_prediction_inputs.py raised "No patients have
# both engineered features and valid outcomes" for the arpi arm on this data
# root. Re-add "arpi" here once that is root-caused.
ENABLED_ARMS = ["adt"]

RUNS = [
    {
        "label": label,
        "title": spec["title"],
        "anchor_col": "none",
        "anchor": spec["anchor"],
        "landmarks": spec["landmarks"],
        "input_csv": DATA_ARPI if spec["anchor"] == "arpi" else DATA_ADT,
        "restrict_to_mrns": spec["mrn_csv"],
        "inputs_dir": SURVIVAL_OUTPUT_ROOT / f"prediction_inputs_{label}",
        "output_dir": SURVIVAL_OUTPUT_ROOT / f"local_runs_{label}",
    }
    for label, spec in COHORT_SPECS.items()
    if label in ENABLED_ARMS
]

os.chdir(PROJECT_ROOT)
MRN_LISTS_DIR.mkdir(parents=True, exist_ok=True)
for run in RUNS:
    run["inputs_dir"].mkdir(parents=True, exist_ok=True)
    run["output_dir"].mkdir(parents=True, exist_ok=True)

for v in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ[v] = "1"

print("python:            ", PYTHON)
print("cwd:               ", os.getcwd())
print("survival_dir:      ", SURVIVAL_DIR)
print("data_preprocessing:", DATA_PREPROCESSING_DIR)
print("output root:       ", NEPC_PROJ_PATH)
print("baseline root:     ", BASELINE_PROJ_PATH, "(read-only, comparison cell)")
print("\nsources:")
for table, path in SOURCES.items():
    exists = "OK     " if path.exists() else "MISSING"
    print(f"  {exists} {table:30s} {path}")
print()
for run in RUNS:
    print(f"{run['label']:30s}: anchor={run['anchor']:4s} landmarks={run['landmarks']} inputs={run['inputs_dir']} outputs={run['output_dir']}")


## 0. Schema audit of the merged parquets

`compile_OncDRS_data.ipynb` inserts any requested-but-absent column as **null**,
printing a warning rather than raising. A column renamed between release pulls
therefore reaches this pipeline as a silently all-null column, and an all-null
`NCI_PREFERRED_MED_NM` or `DERIVED_LAST_ALIVE_DATE` does not crash anything --
it just produces an empty treatment anchor, no platinum events, or no censoring
date, i.e. a plausible-looking but wrong cohort.

This cell checks every column the pipeline reads, in three tiers:

* **required** -- absent or 100% null/blank **raises**.
* **expected** -- absent raises; all-null warns. These are legitimately sparse
  (`HYBRID_DEATH_DT`: most patients are alive; `TEXT_RESULT`: only non-numeric
  lab results populate it).
* **optional** -- absent or all-null warns. The 2nd/3rd ICD-10 code slots and
  the `_NM` name columns.

Non-null fractions are printed for every column, so partial-null cases are
visible even when nothing raises. Worth eyeballing: if the
`PT_INFO_STATUS_REGISTRATION` date columns come back far more null than the row
count suggests, suspect the newest-release-wins dedup
(`DEDUP_COL_MAP[...] = ["DFCI_MRN"]`) letting a sparse release clobber good
rows. That is fixed upstream, not here.


In [ ]:
# Columns each pipeline stage actually reads. Sources:
#   LABS/HEALTH_HISTORY  -> LAB_SCAN_COLUMNS / HEALTH_SCAN_COLUMNS
#                           (longitudinal_data_processing.py)
#   MEDICATIONS          -> MEDICATION_SCAN_COLUMNS (same file)
#   EHR_DIAGNOSES        -> load_and_explode_icd (compile_COMPASS_cohort_data.py)
#   PT_INFO_...          -> load_patient_status (same file)
#
# Three tiers, because "all null" is a bug for some of these columns and
# perfectly normal for others:
#
#   REQUIRED  must be present AND have at least one non-blank value.
#             All-null here means the cohort silently comes out empty or wrong.
#   EXPECTED  must be present; all-null only warns. These are legitimately
#             sparse (HYBRID_DEATH_DT: most patients are alive; TEXT_RESULT:
#             only non-numeric lab results populate it; vitals rows have it
#             explicitly nulled in build_raw_longitudinal_data).
#   OPTIONAL  absent OR all-null only warns. The 2nd/3rd ICD-10 code slots are
#             optional (load_and_explode_icd falls back to a flat one-code-per-
#             row source) and the _NM name columns are read only by the
#             optional compile_MRNs_for_manual_review.py.
REQUIRED_COLUMNS = {
    "EHR_DIAGNOSES": ["DFCI_MRN", "START_DT", "DIAGNOSIS_ICD10_CD"],
    "MEDICATIONS": ["DFCI_MRN", "NCI_PREFERRED_MED_NM", "MED_START_DT"],
    "LABS": [
        "DFCI_MRN", "SPECIMEN_COLLECT_DT", "TEST_TYPE_CD",
        "NUMERIC_RESULT", "RESULT_UOM_NM",
    ],
    "HEALTH_HISTORY": [
        "DFCI_MRN", "CODE_TYPE", "START_DT", "HEALTH_HISTORY_TYPE", "RESULTS",
    ],
    "PT_INFO_STATUS_REGISTRATION": [
        "DFCI_MRN", "BIRTH_DT", "DERIVED_LAST_ALIVE_DATE", "GENDER_NM",
    ],
}

EXPECTED_COLUMNS = {
    # TEST_TYPE_DESCR feeds generate_new_test_name_expr(TEST_TYPE_CD,
    # TEST_TYPE_DESCR); all-null degrades lab naming rather than breaking it.
    "LABS": ["TEST_TYPE_DESCR", "TEXT_RESULT"],
    "HEALTH_HISTORY": ["UNITS_CD"],
    # All-null death dates would mean DEATH == 0 for everyone. The modelled
    # endpoint is platinum, not death, so this warns rather than raising --
    # but it is the loudest thing in this cell for a reason.
    "PT_INFO_STATUS_REGISTRATION": ["HYBRID_DEATH_DT"],
}

OPTIONAL_COLUMNS = {
    "EHR_DIAGNOSES": [
        "DIAGNOSIS_ICD10_CD2", "DIAGNOSIS_ICD10_CD3",
        "DIAGNOSIS_ICD10_NM", "DIAGNOSIS_ICD10_NM2", "DIAGNOSIS_ICD10_NM3",
    ],
}


def nonempty_fraction_expr(name, dtype):
    """Fraction of rows where `name` is neither null nor a blank string.

    Blank matters because the null-fill hazard is not the only way a column
    arrives empty -- a source column that exists but was never populated shows
    up as "" in the CSV releases and survives the merge as an empty string.
    """
    col = pl.col(name)
    if dtype == pl.String:
        return (col.is_not_null() & (col.str.strip_chars() != "")).mean().alias(name)
    return col.is_not_null().mean().alias(name)


problems = []
warnings = []

for table in SOURCE_TABLES:
    path = SOURCES[table]
    print(f"\n===== {table} =====")
    print(f"  path: {path}")
    if not path.exists():
        problems.append(f"{table}: source file not found: {path}")
        print("  MISSING -- skipping")
        continue

    required = REQUIRED_COLUMNS[table]
    expected = EXPECTED_COLUMNS.get(table, [])
    optional = OPTIONAL_COLUMNS.get(table, [])
    tier = {c: t for t, cols in
            (("req", required), ("exp", expected), ("opt", optional))
            for c in cols}

    lf = scan_source(path)
    schema = lf.collect_schema()
    present = list(schema.names())

    # scan_source() casts parquet to all-Utf8, so read dtypes from the parquet
    # itself -- otherwise NUMERIC_RESULT would look like a string column and
    # get the blank-string treatment.
    raw_schema = (
        pl.scan_parquet(path).collect_schema()
        if path.suffix.lower() in (".parquet", ".pq")
        else schema
    )

    checkable = [c for c in required + expected + optional if c in present]
    stats = lf.select(
        [pl.len().alias("__n_rows"), pl.col("DFCI_MRN").n_unique().alias("__n_mrns")]
        + [nonempty_fraction_expr(c, raw_schema[c]) for c in checkable]
    ).collect()

    print(f"  rows: {int(stats['__n_rows'][0]):,}   "
          f"unique DFCI_MRN: {int(stats['__n_mrns'][0]):,}")

    for group, label in ((required, "required"), (expected, "expected")):
        absent = [c for c in group if c not in present]
        if absent:
            problems.append(f"{table}: {label} columns absent: {absent}")
            print(f"  ABSENT ({label}): {absent}")
    absent_optional = [c for c in optional if c not in present]
    if absent_optional:
        warnings.append(f"{table}: optional columns absent: {absent_optional}")
        print(f"  absent (optional): {absent_optional}")

    for c in checkable:
        frac = stats[c][0]
        frac = 0.0 if frac is None else float(frac)
        flag = ""
        if frac == 0.0:
            if tier[c] == "req":
                problems.append(f"{table}.{c}: present but 100% null/blank")
                flag = "   <-- ALL NULL (required)"
            else:
                warnings.append(f"{table}.{c}: present but 100% null/blank")
                flag = f"   <-- all null ({tier[c]})"
        print(f"    {tier[c]} {c:28s} non-null {frac:7.2%}{flag}")

print("\n" + "=" * 70)
for w in warnings:
    print(f"WARN: {w}")
if problems:
    for p in problems:
        print(f"FAIL: {p}")
    raise RuntimeError(
        f"{len(problems)} schema problem(s) in {PROFILE_DATA_ROOT}. Fix COLUMN_MAP / "
        "ALIAS_MAP in PROFILE_data_processing/compile_OncDRS_data.ipynb and re-run "
        "that notebook for the affected tables before continuing."
    )
print(f"Schema audit passed ({len(warnings)} warning(s)): every required column "
      "is present and non-empty.")


## 1. Compile COMPASS cohort data

In [ ]:
stage1_args = " ".join([
    "--icd-source", str(SOURCES["EHR_DIAGNOSES"]),
    "--medications-source", str(SOURCES["MEDICATIONS"]),
    "--patient-status-source", str(SOURCES["PT_INFO_STATUS_REGISTRATION"]),
    "--labs-csv", str(SOURCES["LABS"]),
    "--out-dir", str(NEPC_PROJ_PATH),
    "--mrn-lists-dir", str(MRN_LISTS_DIR),
    "--survival-arms", "adt",
])
print(stage1_args)

!{PYTHON} {DATA_PREPROCESSING_DIR}/compile_COMPASS_cohort_data.py {stage1_args}


## 2. Preprocess raw labs once (longitudinal_data_processing.py)

In [ ]:
# ADT-only source flags. --icd-csv is Stage 1's own output
# (already under the new root), not a raw OncDRS table.
stage2_sources = " ".join([
    "--health-csv", str(SOURCES["HEALTH_HISTORY"]),
    "--labs-csv", str(SOURCES["LABS"]),
    "--medications-csv", str(SOURCES["MEDICATIONS"]),
    "--icd-csv", str(NEPC_PROJ_PATH / "prostate_icd_data.csv"),
])

adt_args = " ".join([
    "--anchor-med-set", "adt",
    "--survival-cohort-csv", str(COHORT_SPECS["adt"]["mrn_csv"]),
    "--output-csv", str(DATA_ADT),
    "--consolidated-cache-parquet", str(CACHE_ADT),
    stage2_sources,
])

# ADT pass: full raw lab standardization is the expensive step in this
# pipeline; the Parquet cache makes reruns cheap, but this first pass may be slow.
print(adt_args)
!{PYTHON} {DATA_PREPROCESSING_DIR}/longitudinal_data_processing.py {adt_args}


## Shared run helpers

In [ ]:
MODEL_TASK_SPECS = [
    ("univariate",   "both",      "cox_agg_univariate_nobs_adjusted.csv"),
    ("elastic-net",  "both",      "cox_agg_multivariable_metrics.csv"),
    ("elastic-net",  "baseline",  "cox_agg_baseline_metrics.csv"),
    ("xgboost",      "both",      "landmark_xgboost_metrics.csv"),
    ("xgboost",      "baseline",  "landmark_xgboost_baseline_metrics.csv"),
]


def tasks_for_run(run):
    """Cross MODEL_TASK_SPECS with this run's own landmark list.

    Both ARPI and ADT arms have landmarks=[0, 90, 180] (15 tasks each).
    """
    return [
        (model, lm, config_dir, metrics_filename)
        for model, config_dir, metrics_filename in MODEL_TASK_SPECS
        for lm in run["landmarks"]
    ]


def model_output_dir(model):
    return "cox" if model in ("univariate", "elastic-net") else "xgboost"


# Output subdir for the shared-canonical-labs univariate arm. This arm is a
# SINGLE invocation over ALL landmarks (univariate_analysis.py --shared-canonical-labs
# --landmark-days <all>): the runner intersects each landmark's canonical labs and
# tests that one shared set at every landmark, so every selected landmark sees an
# identical feature list and their per-lab HRs are directly comparable. Results
# land in cox/landmark_shared/ with landmark_days as a column inside the CSV.
SHARED_UNIVARIATE_DIR = "cox/landmark_shared"
SHARED_UNIVARIATE_FILE = "cox_agg_univariate_nobs_adjusted.csv"


def clear_prediction_inputs(inputs_dir):
    for pattern in (
        "aggregated_landmark*.csv",
        "pre_treatment_lab_long_landmark*.csv",
        "split_assignments_landmark*.csv",
        "gam_trajectory_features_landmark*.csv",
        "gam_fit_diagnostics_landmark*.csv",
    ):
        for p in inputs_dir.glob(pattern):
            p.unlink()
            print(f"  removed {p.name}")
    for fname in (
        "canonical_labs_train_val.csv",
        "build_manifest.json",
        "split_assignments.csv",
        "landmark_mrn_availability.csv",
        "landmark_attrition.json",
    ):
        p = inputs_dir / fname
        if p.exists():
            p.unlink()
            print(f"  removed {p.name}")


def build_prediction_inputs(run):
    print(f"\n========== build inputs: {run['title']} ==========")
    clear_prediction_inputs(run["inputs_dir"])
    cmd = [
        PYTHON, str(DATA_PREPROCESSING_DIR / "build_prediction_inputs.py"),
        "--data", str(run["input_csv"]),
        "--output-dir", str(run["inputs_dir"]),
        "--anchor-col", run["anchor_col"],
        "--landmark-days", *[str(lm) for lm in run["landmarks"]],
        "--time-unit-days", "7",
        "--test-frac", "0.20",
        "--val-frac", "0.20",
        "--min-patient-coverage", "0.20",
    ]
    if run.get("restrict_to_mrns"):
        cmd += ["--restrict-to-mrns", str(run["restrict_to_mrns"])]
    print("[run ] " + " ".join(cmd))
    rc = subprocess.call(cmd)
    if rc != 0:
        raise RuntimeError(f"build_prediction_inputs failed for {run['label']} with rc={rc}")


def cohort_diagnostics(run):
    print(f"\n========== cohort diagnostics: {run['title']} ==========")
    for lm in run["landmarks"]:
        agg_path = run["inputs_dir"] / f"aggregated_landmark{lm}.csv"
        if not agg_path.exists():
            print(f"  landmark +{lm}d: aggregated CSV not found, skipping")
            continue
        agg = pd.read_csv(agg_path)

        def find_col(substr, stat):
            return next(
                (c for c in agg.columns if substr.lower() in c.lower() and c.endswith(f"__{stat}")),
                None,
            )

        n_plat = int(agg["PLATINUM"].sum())
        print(f"=== landmark +{lm}d | n_total={len(agg):,} n_PLATINUM={n_plat} ===")
        for lab_substr in ("Testosterone", "PSA", "Prostate specific Ag"):
            for stat in ("mean", "last", "max", "min"):
                col = find_col(lab_substr, stat)
                if col is None:
                    continue
                for ev in (0, 1):
                    sub = agg.loc[agg["PLATINUM"] == ev, col].dropna()
                    if sub.empty:
                        continue
                    print(
                        f"  {lab_substr:>22s} {stat:5s} PLAT={ev}: median={sub.median():>10.2f} "
                        f"max={sub.max():>12.2f} n={len(sub):>5}"
                    )
                break
        print()


def build_model_command(model, landmark, config_dir, row_output_dir, run):
    if model == "univariate":
        return [
            PYTHON, str(SURVIVAL_DIR / "univariate_analysis.py"),
            "--inputs-dir", str(run["inputs_dir"]),
            "--output-dir", str(row_output_dir),
            "--landmark-days", str(landmark),
            "--endpoints", "platinum",
        ]
    if model == "elastic-net":
        cmd = [
            PYTHON, str(SURVIVAL_DIR / "multivariate_analysis.py"),
            "--model", "elastic-net",
            "--inputs-dir", str(run["inputs_dir"]),
            "--output-dir", str(row_output_dir),
            "--landmark-days", str(landmark),
            "--endpoints", "platinum",
            "--n-folds", str(N_FOLDS),
        ]
        if config_dir == "baseline":
            cmd.append("--baseline")
        return cmd
    if model == "xgboost":
        cmd = [
            PYTHON, str(SURVIVAL_DIR / "multivariate_analysis.py"),
            "--model", "xgboost",
            "--inputs-dir", str(run["inputs_dir"]),
            "--output-dir", str(row_output_dir),
            "--landmark-days", str(landmark),
            "--endpoints", "platinum",
            "--n-folds", str(N_FOLDS),
        ]
        if config_dir == "baseline":
            cmd.append("--baseline")
        return cmd
    raise ValueError(f"Unknown model: {model}")


def run_shared_univariate(run):
    """Univariate arm on ONE shared canonical lab set across all of this run's landmarks.

    Single invocation over every landmark in run["landmarks"] with
    --shared-canonical-labs, so the runner intersects each landmark's canonical
    labs and tests that shared set at every landmark. The resulting CSV carries a
    landmark_days column, so every selected landmark's rows are directly comparable.
    """
    row_output_dir = run["output_dir"] / SHARED_UNIVARIATE_DIR
    row_output_dir.mkdir(parents=True, exist_ok=True)
    metrics_path = row_output_dir / SHARED_UNIVARIATE_FILE
    tag = f"{run['label']:28s} univariate  landmark_shared (labs={','.join(map(str, run['landmarks']))})"
    if metrics_path.exists() and not FORCE_RERUN:
        print(f"[skip] {tag} -> exists")
        return (tag, "skipped", 0.0)
    cmd = [
        PYTHON, str(SURVIVAL_DIR / "univariate_analysis.py"),
        "--inputs-dir", str(run["inputs_dir"]),
        "--output-dir", str(row_output_dir),
        "--landmark-days", *[str(lm) for lm in run["landmarks"]],
        "--endpoints", "platinum",
        "--shared-canonical-labs",
    ]
    print(f"[run ] {tag}")
    print("       " + " ".join(cmd))
    t0 = time.time()
    rc = subprocess.call(cmd)
    elapsed = time.time() - t0
    status = "ok" if rc == 0 else f"FAILED (rc={rc})"
    print(f"[done] {tag} -> {status} ({elapsed/60:.1f} min)\n")
    return (tag, status, elapsed)


def run_models(run):
    print(f"\n========== run models: {run['title']} ==========")
    tasks = tasks_for_run(run)
    summary = []
    for model, landmark, config_dir, metrics_filename in tasks:
        row_output_dir = run["output_dir"] / model_output_dir(model) / f"landmark_{landmark}" / config_dir
        metrics_path = row_output_dir / metrics_filename
        tag = f"{run['label']:28s} {model:11s} landmark_{landmark:<3} {config_dir}"
        if metrics_path.exists() and not FORCE_RERUN:
            print(f"[skip] {tag} -> {metrics_path.relative_to(run['output_dir'])} exists")
            summary.append((tag, "skipped", 0.0))
            continue
        row_output_dir.mkdir(parents=True, exist_ok=True)
        cmd = build_model_command(model, landmark, config_dir, row_output_dir, run)
        print(f"[run ] {tag}")
        print("       " + " ".join(cmd))
        t0 = time.time()
        rc = subprocess.call(cmd)
        elapsed = time.time() - t0
        status = "ok" if rc == 0 else f"FAILED (rc={rc})"
        print(f"[done] {tag} -> {status} ({elapsed/60:.1f} min)\n")
        summary.append((tag, status, elapsed))
    # Shared-canonical-labs univariate arm (single run over all of this run's landmarks).
    summary.append(run_shared_univariate(run))
    print("\n=== run summary ===")
    for tag, status, elapsed in summary:
        print(f"  {tag} {status:>20s} {elapsed/60:6.1f} min")
    return summary


def summarize_outputs(run):
    rows = []
    for model, landmark, config_dir, metrics_filename in tasks_for_run(run):
        if model == "univariate":
            continue
        metrics_path = run["output_dir"] / model_output_dir(model) / f"landmark_{landmark}" / config_dir / metrics_filename
        base = {"run": run["label"], "model": model, "landmark": landmark, "config": config_dir, "endpoint": "platinum"}
        if not metrics_path.exists():
            rows.append({**base, "n_test": None, "n_test_events": None, "c_index": None,
                         "mean_auc_t": None, "integrated_brier": None, "status": "missing"})
            continue
        df = pd.read_csv(metrics_path)
        platinum = df.loc[df["endpoint"] == "platinum"]
        if platinum.empty:
            rows.append({**base, "n_test": None, "n_test_events": None, "c_index": None,
                         "mean_auc_t": None, "integrated_brier": None, "status": "no platinum row"})
            continue
        platinum = platinum.iloc[0]
        if model == "elastic-net":
            rows.append({
                **base,
                "n_test": int(platinum["n_test"]),
                "n_test_events": int(platinum["n_events_test"]),
                "c_index": float(platinum["test_c_index"]),
                "mean_auc_t": float(platinum["test_mean_auc_t"]),
                "integrated_brier": float(platinum["test_integrated_brier"]),
                "status": "ok",
            })
        elif model == "xgboost":
            rows.append({
                **base,
                "n_test": int(platinum["n_test"]),
                "n_test_events": int(platinum["n_test_events"]),
                "c_index": float(platinum["c_index"]),
                "mean_auc_t": float(platinum["mean_auc_t"]),
                "integrated_brier": float(platinum["integrated_brier"]),
                "status": "ok",
            })
    return pd.DataFrame(rows).sort_values(["run", "landmark", "model", "config"]).reset_index(drop=True)


def summarize_univariate_shared(run):
    """Per-lab platinum associations at every landmark for this run, on the SHARED lab set.

    Reads the shared-arm univariate CSV (identical canonical lab set at every
    landmark, landmark_days as a column) so PSA/testosterone HRs can be read
    across landmarks side by side.
    """
    path = run["output_dir"] / SHARED_UNIVARIATE_DIR / SHARED_UNIVARIATE_FILE
    if not path.exists():
        print(f"  shared univariate output missing: {path}")
        return pd.DataFrame()
    df = pd.read_csv(path)
    df = df.loc[df["endpoint"] == "platinum"].copy()
    keep = ["landmark_days", "feature", "lab_name", "feature_stat",
            "hazard_ratio_per_sd", "p_value", "q_value", "n_patients_used", "n_events_used"]
    keep = [c for c in keep if c in df.columns]
    sort_cols = [c for c in ("lab_name", "feature_stat", "landmark_days") if c in df.columns]
    return df[keep].sort_values(sort_cols).reset_index(drop=True)


## 3. Run the ADT cohort

Runs input construction -> GAM trajectory features -> diagnostics -> Python models -> GAM Cox nonlinearity -> summaries for the ADT arm. Set `RUN_GAM = False` in shared setup to retain the pre-GAM workflow. Results are collected in `run_summaries["adt"]` and `summary_dfs["adt"]`.


In [ ]:
from COMPASS.survival_analysis.gam_runner import (
    run_gam_cox_nonlinearity,
    run_gam_trajectory_features,
)

run_summaries = {}
summary_dfs = {}

for run in RUNS:
    build_prediction_inputs(run)
    if RUN_GAM:
        run_gam_trajectory_features(
            inputs_dir=run["inputs_dir"],
            landmark_days=run["landmarks"],
            force=FORCE_RERUN,
        )
    cohort_diagnostics(run)
    run_summaries[run["label"]] = run_models(run)
    if RUN_GAM:
        run_gam_cox_nonlinearity(
            inputs_dir=run["inputs_dir"],
            model_output_dir=run["output_dir"],
            landmark_days=run["landmarks"],
            force=FORCE_RERUN,
        )
    summary_dfs[run["label"]] = summarize_outputs(run)

summary_dfs[RUNS[0]["label"]]


### Per-cohort summary tables

Index into `summary_dfs["<label>"]` for either arm, e.g.
`summary_dfs["arpi"]` or `summary_dfs["adt"]`. No other cohort labels are
populated.


## 4. Summary


In [ ]:
combined_summary_df = pd.concat(summary_dfs.values(), ignore_index=True)
combined_summary_df


## 5. Cohort comparison vs. the `ALL_2025_03` run

Per-stage patient counts for this run against `data/CAIA/COMPASS/`. Reads only;
nothing under the baseline root is written.

What to expect, and what would mean something is wrong:

* **Stage 1 cohort should grow.** Seven merged releases contribute more C61
  patients and more ADT starts than one. A flat or shrinking `stage1_n_cohort`
  means the ICD or medications source did not actually widen.
* **`stage1_n_platinum_dated` must be non-zero and should grow.** Near-zero is
  the signature of an all-null `NCI_PREFERRED_MED_NM` -- go back to the schema
  audit and the upstream `COLUMN_MAP`.
* **Landmark counts can move in both directions.** `LABS` and `MEDICATIONS` are
  exact-full-row deduplicated upstream, so genuinely repeated lab draws sharing
  patient + timestamp + value + units collapse to one row. The >=5-broad-PSA
  gate counts rows, so a few patients can fall *below* threshold even as the
  overall cohort grows. A large drop, though, is worth investigating.
* `ALL_2026_03` contributes no `LABS` table at all, so lab coverage gains come
  from the older releases only.


In [ ]:
def read_landmark_attrition(root, label):
    path = root / "survival_analysis" / f"prediction_inputs_{label}" / "landmark_attrition.json"
    if not path.exists():
        return None
    return json.loads(path.read_text())


def read_cohort_csv_counts(root, label):
    """Stage 1 counts straight from that arm's survival cohort CSV."""
    path = root / f"prostate_{label}_survival_cohort_{label}.csv"
    if not path.exists():
        return {}
    df = pd.read_csv(path, usecols=lambda c: c in ("DFCI_MRN", "PLATINUM_DATE", "DEATH"))
    out = {"stage1_n_cohort": int(df["DFCI_MRN"].nunique())}
    if "PLATINUM_DATE" in df.columns:
        out["stage1_n_platinum_dated"] = int(df["PLATINUM_DATE"].notna().sum())
    if "DEATH" in df.columns:
        out["stage1_n_deaths"] = int(df["DEATH"].sum())
    return out


def read_landmark_counts(root, label, landmarks):
    inputs_dir = root / "survival_analysis" / f"prediction_inputs_{label}"
    out = {}
    attrition = read_landmark_attrition(root, label)
    if attrition:
        out["stage3_n_loaded_cohort"] = attrition.get("n_loaded_cohort")
        for lm, n in (attrition.get("eligible_by_landmark") or {}).items():
            out[f"lm{lm}_n_eligible"] = n
    for lm in landmarks:
        agg_path = inputs_dir / f"aggregated_landmark{lm}.csv"
        if not agg_path.exists():
            continue
        agg = pd.read_csv(agg_path, usecols=["DFCI_MRN", "PLATINUM"])
        out[f"lm{lm}_n_total"] = int(len(agg))
        out[f"lm{lm}_n_platinum"] = int(agg["PLATINUM"].sum())
    return out


def collect_metrics(root, label, landmarks):
    m = {}
    m.update(read_cohort_csv_counts(root, label))
    m.update(read_landmark_counts(root, label, landmarks))
    return m


comparison_rows = []
for run in RUNS:
    label, landmarks = run["label"], run["landmarks"]
    new = collect_metrics(NEPC_PROJ_PATH, label, landmarks)
    old = collect_metrics(BASELINE_PROJ_PATH, label, landmarks)
    for metric in sorted(set(new) | set(old), key=lambda k: (k.split("_")[0], k)):
        a, b = old.get(metric), new.get(metric)
        comparison_rows.append({
            "arm": label,
            "metric": metric,
            "ALL_2025_03": a,
            "PROFILE_DATA": b,
            "delta": (b - a) if (a is not None and b is not None) else None,
            "pct_change": (100.0 * (b - a) / a) if (a not in (None, 0) and b is not None) else None,
        })

comparison_df = pd.DataFrame(comparison_rows)
if comparison_df.empty:
    print("Nothing to compare yet -- run the stages above (and confirm the "
          f"baseline run exists at {BASELINE_PROJ_PATH}).")
else:
    missing_baseline = comparison_df["ALL_2025_03"].isna().all()
    if missing_baseline:
        print(f"WARNING: no baseline outputs found under {BASELINE_PROJ_PATH}; "
              "deltas are unavailable.")
    with pd.option_context("display.max_rows", 200, "display.width", 160):
        print(comparison_df.to_string(index=False, float_format=lambda v: f"{v:,.1f}"))

# Stage 2 attrition is written per OUTPUT ROOT, not per arm
# (longitudinal_data_processing.py writes cohort_attrition.json next to
# --output-csv, and both arms share that directory), so it reflects whichever
# anchor ran last -- ADT above. Shown separately for that reason.
print("\n=== Stage 2 cohort_attrition.json (last anchor run in each root: adt) ===")
for name, root in (("ALL_2025_03", BASELINE_PROJ_PATH), ("PROFILE_DATA", NEPC_PROJ_PATH)):
    path = root / "cohort_attrition.json"
    if not path.exists():
        print(f"  {name}: not found at {path}")
        continue
    attrition = json.loads(path.read_text())
    print(f"  {name}: n_output_patients={attrition.get('n_output_patients')} "
          f"n_with_highlighted_treatment_anchor={attrition.get('n_with_highlighted_treatment_anchor')} "
          f"n_after_broad_icd_filter={attrition.get('n_after_broad_icd_filter')}")

comparison_df
